In [ ]:
# Imports
import sys
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as scipy_signal
from scipy.interpolate import interp1d
from pathlib import Path

# Ensure project root is on sys.path so `src.*` imports work from inside nested notebook dirs
def find_project_root(start: Path = None, marker: str = 'src') -> Path | None:
	if start is None:
		start = Path.cwd()
	p = start.resolve()
	for cand in [p] + list(p.parents):
		if (cand / marker).exists():
			return cand
	return None

root = find_project_root()
if root is None:
	raise RuntimeError("Could not locate project root containing 'src' directory")

sys.path.insert(0, str(root))

from src.feature_extraction import HRVExtractor
from src.preprocessing import ECGPreprocessor
from src.visualization import SignalVisualizer, AnalysisVisualizer

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
print('Imports successful.')

ModuleNotFoundError: No module named 'src.feature_extraction'

In [ ]:
# Configuration
SAMPLING_RATE  = 500
RESULTS_DIR    = Path('../results/plots')
DATA_PROCESSED = Path('../data/processed')
DB_PATH        = DATA_PROCESSED / 'neuro_genomic.db'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load separated signals (priority: database -> CSV -> synthetic fallback)
loaded_from = None

# 1) Database first
if DB_PATH.exists():
    try:
        with sqlite3.connect(DB_PATH) as conn:
            df = pd.read_sql_query('SELECT * FROM separated_components', conn)
        if {'time_s', 'maternal_ecg', 'fetal_ecg'}.issubset(df.columns):
            t = df['time_s'].values
            maternal_ecg = df['maternal_ecg'].values
            fetal_ecg = df['fetal_ecg'].values
            loaded_from = f'database ({DB_PATH})'
    except Exception:
        pass

# 2) CSV fallback
if loaded_from is None:
    sep_path = DATA_PROCESSED / 'separated_components.csv'
    if sep_path.exists():
        df = pd.read_csv(sep_path)
        t = df['time_s'].values
        maternal_ecg = df['maternal_ecg'].values
        fetal_ecg = df['fetal_ecg'].values
        loaded_from = f'csv ({sep_path})'

# 3) Synthetic fallback
if loaded_from is None:
    DURATION = 10
    t = np.linspace(0, DURATION, SAMPLING_RATE * DURATION)
    np.random.seed(42)

    def make_ecg(t, bpm, amp=1.0):
        rr = 60.0 / bpm
        sig = np.zeros_like(t)
        for bt in np.arange(0, t[-1], rr):
            sig += amp * np.exp(-((t - bt) ** 2) / (2 * 0.02 ** 2))
        return sig + 0.03 * np.random.randn(len(t))

    maternal_ecg = make_ecg(t, bpm=75)
    fetal_ecg = make_ecg(t, bpm=145, amp=0.5)
    loaded_from = 'synthetic fallback'

print(f'Loaded source: {loaded_from}')
print(f'Maternal: {maternal_ecg.shape}  Fetal: {fetal_ecg.shape}')

In [ ]:
# R-peak detection
extractor_maternal = HRVExtractor(sampling_rate=SAMPLING_RATE)
extractor_fetal    = HRVExtractor(sampling_rate=SAMPLING_RATE)

maternal_peaks = extractor_maternal.detect_r_peaks(maternal_ecg)
fetal_peaks    = extractor_fetal.detect_r_peaks(fetal_ecg)

hr_maternal = 60 / (np.mean(np.diff(maternal_peaks)) / SAMPLING_RATE)
hr_fetal    = 60 / (np.mean(np.diff(fetal_peaks))    / SAMPLING_RATE)
print(f'Maternal: {len(maternal_peaks)} peaks, ~{hr_maternal:.1f} bpm')
print(f'Fetal:    {len(fetal_peaks)} peaks, ~{hr_fetal:.1f} bpm')

In [ ]:
# Plot R-peaks
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, sig, peaks, label, color in zip(
        axes, [maternal_ecg, fetal_ecg], [maternal_peaks, fetal_peaks],
        ['Maternal', 'Fetal'], ['steelblue', 'tomato']):
    ax.plot(t, sig, linewidth=0.7, color=color, label='ECG')
    ax.plot(t[peaks], sig[peaks], 'x', color='black', ms=6, label='R-peaks')
    ax.set_title(f'{label} R-Peak Detection'); ax.set_ylabel('Amplitude')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nb03_01_r_peaks.png', bbox_inches='tight')
plt.show()

In [ ]:
# Time-domain HRV features
maternal_features = extractor_maternal.extract_features(maternal_ecg)
fetal_features    = extractor_fetal.extract_features(fetal_ecg)

display_keys = ['num_beats', 'heart_rate_mean', 'heart_rate_std',
                'rr_interval_mean', 'rr_interval_std', 'rmssd', 'pnn50']

print(f'{"Feature":<22}  {"Maternal":>12}  {"Fetal":>12}')
print('-' * 50)
for k in display_keys:
    mv = maternal_features.get(k, float('nan'))
    fv = fetal_features.get(k, float('nan'))
    if isinstance(mv, (int, float, np.floating)):
        print(f'{k:<22}  {mv:>12.3f}  {fv:>12.3f}')

In [ ]:
# RR-interval tachograms
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
for ax, feats, label, color in zip(
        axes, [maternal_features, fetal_features], ['Maternal', 'Fetal'], ['steelblue', 'tomato']):
    rr = feats['rr_intervals']
    if len(rr) > 0:
        ax.plot(rr, 'o-', color=color, ms=4, linewidth=0.9)
        ax.axhline(np.mean(rr), color='black', ls='--', lw=0.8, label=f'Mean: {np.mean(rr):.1f} ms')
        ax.set_title(f'{label} RR-Interval Tachogram'); ax.set_ylabel('RR (ms)')
        ax.set_xlabel('Beat index'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nb03_02_tachograms.png', bbox_inches='tight')
plt.show()

In [ ]:
# Frequency-domain HRV (LF, HF, LF/HF ratio via Welch PSD)
def frequency_domain_hrv(peaks, fs, interp_hz=4.0):
    if len(peaks) < 4:
        return {'lf': np.nan, 'hf': np.nan, 'lf_hf_ratio': np.nan}

    rr_s = np.diff(peaks) / fs
    t_rr = peaks[1:] / fs

    # Cubic interpolation to uniform grid
    interp_fn = interp1d(t_rr, rr_s, kind='cubic', fill_value='extrapolate')
    t_uni  = np.arange(t_rr[0], t_rr[-1], 1.0 / interp_hz)
    rr_uni = interp_fn(t_uni)

    freqs, psd = scipy_signal.welch(rr_uni, fs=interp_hz, nperseg=min(256, len(rr_uni)))

    def band_power(f, p, flo, fhi):
        mask = (f >= flo) & (f < fhi)
        return np.trapz(p[mask], f[mask])

    lf = band_power(freqs, psd, 0.04, 0.15)
    hf = band_power(freqs, psd, 0.15, 0.40)
    return {'lf': lf, 'hf': hf, 'lf_hf_ratio': lf / hf if hf > 0 else np.nan,
            'freqs': freqs, 'psd': psd}

freq_maternal = frequency_domain_hrv(maternal_peaks, SAMPLING_RATE)
freq_fetal    = frequency_domain_hrv(fetal_peaks,    SAMPLING_RATE)

for key in ('lf', 'hf', 'lf_hf_ratio'):
    print(f'  {key:<14}: mat={freq_maternal[key]:.4f}  fet={freq_fetal[key]:.4f}')

In [ ]:
# HRV power spectra (LF / HF bands)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
band_colors = {'VLF': '#aec6cf', 'LF': '#ffb347', 'HF': '#77dd77'}
for ax, fd, label in zip(axes, [freq_maternal, freq_fetal], ['Maternal', 'Fetal']):
    if 'freqs' not in fd:
        ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center'); continue
    ax.semilogy(fd['freqs'], fd['psd'], color='navy', linewidth=1)
    ax.axvspan(0.003, 0.04, alpha=0.2, color=band_colors['VLF'], label='VLF')
    ax.axvspan(0.04,  0.15, alpha=0.3, color=band_colors['LF'],  label=f'LF={fd["lf"]:.2e}')
    ax.axvspan(0.15,  0.40, alpha=0.3, color=band_colors['HF'],  label=f'HF={fd["hf"]:.2e}')
    ax.set_xlim([0, 0.5]); ax.set_title(f'{label} HRV PSD')
    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (s²/Hz)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'nb03_03_hrv_power_spectra.png', bbox_inches='tight')
plt.show()

In [ ]:
# Assemble feature matrix (time + frequency domain)
def build_feature_row(time_domain, freq_domain, prefix):
    scalar_keys = ['num_beats', 'heart_rate_mean', 'heart_rate_std',
                   'rr_interval_mean', 'rr_interval_std', 'rmssd', 'pnn50']
    row = {f'{prefix}_{k}': time_domain[k] for k in scalar_keys}
    for k in ('lf', 'hf', 'lf_hf_ratio'):
        row[f'{prefix}_{k}'] = freq_domain.get(k, np.nan)
    return row

feature_row = {}
feature_row.update(build_feature_row(maternal_features, freq_maternal, 'mat'))
feature_row.update(build_feature_row(fetal_features,    freq_fetal,    'fet'))

df_features = pd.DataFrame([feature_row])
print(f'Feature matrix: {df_features.shape}')
print(df_features.T.rename(columns={0: 'value'}))

In [ ]:
# Save feature matrix to CSV + SQLite database
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
feat_path = DATA_PROCESSED / 'hrv_feature_matrix.csv'
df_features.to_csv(feat_path, index=False)

with sqlite3.connect(DB_PATH) as conn:
    df_features.to_sql('hrv_feature_matrix', conn, if_exists='replace', index=False)

print(f'Saved CSV: {feat_path}')
print(f'Saved DB table: hrv_feature_matrix in {DB_PATH}')

In [ ]:
# Feature correlation heatmap (simulated multi-sample demo)
np.random.seed(0)
demo_rows = []
for _ in range(30):
    row = {k: v + np.random.randn() * 0.05 * abs(v + 1e-3) for k, v in feature_row.items()}
    demo_rows.append(row)

df_demo = pd.DataFrame(demo_rows)
numeric_cols = df_demo.select_dtypes(include=np.number).columns

viz = AnalysisVisualizer()
fig = viz.plot_correlation_matrix(df_demo[numeric_cols], figsize=(12, 10))
fig.savefig(RESULTS_DIR / 'nb03_04_feature_correlation.png', bbox_inches='tight')
plt.show()

NameError: name 'np' is not defined